In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Created on 2024-05-16

@author: Juan Enrique López Marcos

@description: Jupyter Notebook creado para obtener las técnicas a partir de los catálogos de reglas de detección (Sentinel, Splunk y QRadar) 

"""

**Requerimientos**

In [4]:
import csv
import os
import pandas as pd
from attackcti import attack_client

**Funciones y cliente**

In [ ]:
lift = attack_client()

In [ ]:
def techniques(lift):
    #Solicitud a la libreria attackcti la extraccion de las tecnicas Enterprise y normalizar el json en un dataframe
    techniques = lift.get_enterprise_techniques(stix_format=False)
    techniques = pd.json_normalize(techniques)
    #Eliminar las tecnicas deprecadas y revocadas
    techniques = techniques[(techniques['mitre_deprecated'] != True)]
    # Eliminamos duplicados y convertimos en lista
    techniques = techniques['technique_id'].drop_duplicates().tolist()
    return techniques

In [ ]:
# Buscamos en la columna de descripción
def find_techniques(texto):
    if isinstance(texto, str):
        found = []
        for item in techniques_enterprise:
            if item in texto:
                found.append(item)
        return ', '.join(found)
    else:
        return ''

In [ ]:
def techniques_info(lift):
    techniques_info = lift.get_enterprise_techniques(stix_format=False)
    techniques_info = pd.json_normalize(techniques_info)
    #Eliminar las tecnicas deprecadas y revocadas
    techniques_info = techniques_info[(techniques_info['mitre_deprecated'] != True)]
    techniques_info = techniques_info[['technique_id','type', 'url', 'technique', 'technique_description','tactic','data_sources']]
    return techniques_info

**Inputs**

In [99]:
# path_UCMCatalog2024 = r'C:\Users\jelopez\Documents\CyberProof\python\check_0905\UCM Catalog 2024' OLD
path_UCMCatalog2024 = os.path.join(os.getcwd(), 'UCM Catalog 2024')

path_UCMCatalog2024_Sentinel = path_UCMCatalog2024 + '/UCM Catalog 2024 [Sentinel].xlsx'
path_UCMCatalog2024_Qradar = path_UCMCatalog2024 + '/UCM Catalog 2024 [Qradar].xlsx'
path_UCMCatalog2024_Splunk = path_UCMCatalog2024 + '/UCM Catalog 2024 [Splunk].xlsx'

**Parámetros**

In [102]:
just_id = False

In [8]:
techniques_enterprise = techniques(lift)
techniques_enterprise

### 1. UCM Catalog 2024 - SENTINEL

In [139]:
UCMCatalog2024_Sentinel = pd.read_excel(path_UCMCatalog2024_Sentinel)
UCMCatalog2024_Sentinel['techniques'] = UCMCatalog2024_Sentinel['techniques'].str.replace('[','')
UCMCatalog2024_Sentinel['techniques'] = UCMCatalog2024_Sentinel['techniques'].str.replace(']','')
UCMCatalog2024_Sentinel['techniques'] = UCMCatalog2024_Sentinel['techniques'].str.replace('"','')
UCMCatalog2024_Sentinel['techniques'] = UCMCatalog2024_Sentinel['techniques'].str.split(',')
UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel.explode('techniques').reset_index()
UCMCatalog2024_Sentinel['QUERY'] = UCMCatalog2024_Sentinel['query'].str.upper()
UCMCatalog2024_Sentinel['QUERY'] = UCMCatalog2024_Sentinel['QUERY'].fillna('N/A')
UCMCatalog2024_Sentinel.head(2)

,index,displayName,tactics,techniques,query,QUERY
0,0,Azure Redis Cache - Config modification attempt,"[""Execution""]",T0863,AzureActivity\n| where OperationNameValue cont...,AZUREACTIVITY\n| WHERE OPERATIONNAMEVALUE CONT...
1,1,Azure Redis Cache - Access keys regeneration a...,"[""ResourceDevelopment""]",T1587,AzureActivity\n| where OperationName == 'Regen...,AZUREACTIVITY\n| WHERE OPERATIONNAME == 'REGEN...


In [140]:
if just_id:
    UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel.assign(type_source='item raw')[['techniques', 'type_source']]
    UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel_raw[UCMCatalog2024_Sentinel_raw['techniques']!='']
    UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel_raw.dropna(subset=['techniques'])
    UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel_raw.drop_duplicates()
    UCMCatalog2024_Sentinel_raw
else:
    UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel.assign(type_source='item raw')[['techniques', 'query','type_source']]
    UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel_raw[UCMCatalog2024_Sentinel_raw['techniques']!='']
    UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel_raw.dropna(subset=['techniques'])
    UCMCatalog2024_Sentinel_raw = UCMCatalog2024_Sentinel_raw.drop_duplicates()
    UCMCatalog2024_Sentinel_raw

In [141]:
UCMCatalog2024_Sentinel_raw

,techniques,query,type_source
0,T0863,AzureActivity\n| where OperationNameValue cont...,item raw
1,T1587,AzureActivity\n| where OperationName == 'Regen...,item raw
4,T1566,CrossSentinelQuery_SigninLogs_OpCanada(1h)\n| ...,item raw
5,T1562,AzureDevOpsAuditing\n| where OperationName =~ ...,item raw
7,T1078,AzureDevOpsAuditing\n| where OperationName =~ ...,item raw
...,...,...,...
3786,T1203,// UC383 - WAF - Blocked Command Injection fro...,item raw
3787,T1203,// UC384 - WAF - Blocked SQL Injection from In...,item raw
3788,T1011,// UC438 - WAF - Blind SQL Injection from Inte...,item raw
3789,T1011,// UC439 - WAF - Command Injection from Intern...,item raw


In [8]:
# Set mantenido por si acaso
ids_UCMCatalog2024_Sentinel_raw = set(UCMCatalog2024_Sentinel['techniques'])

In [10]:
if just_id:
    UCMCatalog2024_Sentinel['techniques_from_query'] = UCMCatalog2024_Sentinel['query'].apply(lambda x: find_techniques(x))
    UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel[UCMCatalog2024_Sentinel['techniques_from_query']!='']
    UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel.dropna(subset=['techniques_from_query'])
    UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel['techniques_from_query'].str.split(',').explode('techniques_from_query').reset_index()
else:
    UCMCatalog2024_Sentinel['techniques_from_query'] = UCMCatalog2024_Sentinel['query'].apply(lambda x: find_techniques(x))
    UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel[UCMCatalog2024_Sentinel['techniques_from_query']!='']
    UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel.dropna(subset=['techniques_from_query'])
    UCMCatalog2024_Sentinel['techniques_from_query_explode'] = UCMCatalog2024_Sentinel['techniques_from_query'].str.split(',')
    UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel.explode('techniques_from_query')
    UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel.drop_duplicates()
    UCMCatalog2024_Sentinel = UCMCatalog2024_Sentinel.reset_index(drop=True)
    UCMCatalog2024_Sentinel.head(2)

,index,techniques_from_query
0,0,T1597.001
1,1,T1597


In [154]:
UCMCatalog2024_Sentinel_cs

,index,displayName,tactics,techniques,query,QUERY,techniques_from_query,techniques_from_query_explode
366,325,SOC - FW - Threat Intelligence - Malicious IP,"[""InitialAccess""]",,// Threat Intelligence - Malicious IP\n// Mitr...,// THREAT INTELLIGENCE - MALICIOUS IP\n// MITR...,"T1597.001, T1597",T1597.001
366,325,SOC - FW - Threat Intelligence - Malicious IP,"[""InitialAccess""]",,// Threat Intelligence - Malicious IP\n// Mitr...,// THREAT INTELLIGENCE - MALICIOUS IP\n// MITR...,"T1597.001, T1597",T1597
367,326,SOC - FW - Inbound Traffic on Sensitive Ports,"[""InitialAccess""]",,// Inbound Traffic on Sensitive Ports\n// Mitr...,// INBOUND TRAFFIC ON SENSITIVE PORTS\n// MITR...,T1133,T1133
368,327,SOC - FW - Internal SMB Scan,"[""InitialAccess""]",,// Internal SMB Scan\n// Mitre Tactic: TA0007\...,// INTERNAL SMB SCAN\n// MITRE TACTIC: TA0007\...,T1135,T1135
394,352,SOC - AD Possible Kerberoasting,"[""CredentialAccess""]",T1558,// T1558 - Kerberoasting\n// Reference: https:...,// T1558 - KERBEROASTING\n// REFERENCE: HTTPS:...,T1558,T1558
...,...,...,...,...,...,...,...,...
3786,2881,UC383 - WAF - Blocked Command Injection from I...,['Execution'],T1203,// UC383 - WAF - Blocked Command Injection fro...,// UC383 - WAF - BLOCKED COMMAND INJECTION FRO...,T1203,T1203
3787,2882,UC384 - WAF - Blocked SQL Injection from Inter...,['Execution'],T1203,// UC384 - WAF - Blocked SQL Injection from In...,// UC384 - WAF - BLOCKED SQL INJECTION FROM IN...,T1203,T1203
3788,2883,UC438 - WAF - Blind SQL Injection from Interna...,[],T1011,// UC438 - WAF - Blind SQL Injection from Inte...,// UC438 - WAF - BLIND SQL INJECTION FROM INTE...,T1011,T1011
3789,2884,UC439 - WAF - Command Injection from Internal ...,"['Exfiltration', 'Persistence']",T1011,// UC439 - WAF - Command Injection from Intern...,// UC439 - WAF - COMMAND INJECTION FROM INTERN...,T1011,T1011


In [11]:
if just_id:
    UCMCatalog2024_Sentinel_description = UCMCatalog2024_Sentinel.assign(type_source='item description')[['techniques_from_query', 'type_source']]
    UCMCatalog2024_Sentinel_description = UCMCatalog2024_Sentinel_description[UCMCatalog2024_Sentinel_description['techniques_from_query']!='']
    UCMCatalog2024_Sentinel_description.rename(columns={'techniques_from_query': 'techniques'}, inplace=True)
    UCMCatalog2024_Sentinel_description = UCMCatalog2024_Sentinel_description.drop_duplicates()
    UCMCatalog2024_Sentinel_description.head(2)
else:
    

,techniques,type_source
0,T1597.001,item description
1,T1597,item description


In [164]:
UCMCatalog2024_Sentinel_description = UCMCatalog2024_Sentinel.assign(type_source='item description')[['techniques_from_query_explode','query', 'type_source']]
UCMCatalog2024_Sentinel_description.rename(columns={'techniques_from_query_explode': 'techniques'}, inplace=True)
UCMCatalog2024_Sentinel_description

,techniques,query,type_source
366,T1597.001,// Threat Intelligence - Malicious IP\n// Mitr...,item description
366,T1597,// Threat Intelligence - Malicious IP\n// Mitr...,item description
367,T1133,// Inbound Traffic on Sensitive Ports\n// Mitr...,item description
368,T1135,// Internal SMB Scan\n// Mitre Tactic: TA0007\...,item description
394,T1558,// T1558 - Kerberoasting\n// Reference: https:...,item description
...,...,...,...
3786,T1203,// UC383 - WAF - Blocked Command Injection fro...,item description
3787,T1203,// UC384 - WAF - Blocked SQL Injection from In...,item description
3788,T1011,// UC438 - WAF - Blind SQL Injection from Inte...,item description
3789,T1011,// UC439 - WAF - Command Injection from Intern...,item description


In [165]:
# Unimos técnicas de UCMCatalog2024 Sentinel
UCMCatalog2024_Sentinel_union = pd.concat([UCMCatalog2024_Sentinel_raw, UCMCatalog2024_Sentinel_description])
UCMCatalog2024_Sentinel_union['techniques'] = UCMCatalog2024_Sentinel_union['techniques'].str.strip()
UCMCatalog2024_Sentinel_union = UCMCatalog2024_Sentinel_union.drop_duplicates(subset=['techniques'])
UCMCatalog2024_Sentinel_union = UCMCatalog2024_Sentinel_union.sort_values(by='techniques')
UCMCatalog2024_Sentinel_union.head()


,techniques,query,type_source
3483,T0802,CommonSecurityLog\n | where DeviceVendor in...,item raw
777,T0807,// Remove items from the artifacts list in ord...,item raw
527,T0813,let TimeRange = 1h;\nCLCCEF_CL\n| extend inges...,item raw
929,T0814,CLCFortinet_CL\n| where isnotempty(Message)\n|...,item raw
925,T0815,CLCFortinet_CL\n| where isnotempty(Message)\n|...,item raw


In [166]:
UCMCatalog2024_Sentinel_union.groupby('type_source').count()

,techniques,query
type_source,,
item description,3,3
item raw,221,221


In [167]:
# Guardado del fichero final
UCMCatalog2024_Sentinel_union.to_csv('tecnhniques_UCMCatalog2024_Sentinel_withquery.csv',sep=';',encoding='utf8',index=False)

### 2. UCM Catalog 2024 - SPLUNK

In [15]:
UCMCatalog2024_Splunk = pd.read_excel(path_UCMCatalog2024_Splunk)
UCMCatalog2024_Splunk.head(3)

,title,description,search,severity,security_domain,MITRE Tactic,Mitre Technique
0,Access - Abstraction Non-Us Login - Rule,Abstractors are not allowed to work within Pat...,index=duo sourcetype!=okta:im p- user=p-* fact...,high,access,NaN,NaN
1,Access - Credential Stuffing - Rule,Detects automated injection of stolen username...,| tstats summariesonly=t count as actioncount ...,low,access,NaN,NaN
2,Access - Drug Cabinet Superuser usage - Rule,Alerts when superuser credentials are used to ...,index=oelogs sourcetype=oe_application_logs On...,high,access,NaN,NaN


In [16]:
UCMCatalog2024_Splunk['MITRE_TECHNIQUE'] = UCMCatalog2024_Splunk['Mitre Technique'].str.upper()
UCMCatalog2024_Splunk['DESCRIPTION'] = UCMCatalog2024_Splunk['description'].str.upper()
UCMCatalog2024_Splunk['SEARCH'] = UCMCatalog2024_Splunk['search'].str.upper()

In [18]:
# Obtención técnicas a partir de una columna dada
UCMCatalog2024_Splunk_raw = UCMCatalog2024_Splunk
UCMCatalog2024_Splunk_raw['techniques_from_raw'] = UCMCatalog2024_Splunk_raw['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Splunk_raw = UCMCatalog2024_Splunk_raw[UCMCatalog2024_Splunk_raw['techniques_from_raw']!='']
UCMCatalog2024_Splunk_raw = UCMCatalog2024_Splunk_raw.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Splunk_raw = UCMCatalog2024_Splunk_raw['techniques_from_raw'].str.split(',').explode('techniques_from_raw').reset_index()
UCMCatalog2024_Splunk_raw.head(2)

,index,techniques_from_raw
0,0,T1531
1,1,T1098


In [19]:
UCMCatalog2024_Splunk_raw.shape

(105, 2)

In [20]:
# Formateo para unión final
UCMCatalog2024_Splunk_raw = UCMCatalog2024_Splunk_raw.assign(type_source='item raw')[['techniques_from_raw', 'type_source']]
UCMCatalog2024_Splunk_raw = UCMCatalog2024_Splunk_raw[UCMCatalog2024_Splunk_raw['techniques_from_raw']!='']
UCMCatalog2024_Splunk_raw.rename(columns={'techniques_from_raw': 'techniques'}, inplace=True)
UCMCatalog2024_Splunk_raw = UCMCatalog2024_Splunk_raw.drop_duplicates()
UCMCatalog2024_Splunk_raw.head(2)

,techniques,type_source
0,T1531,item raw
1,T1098,item raw


In [21]:
# Obtención técnicas a partir de una columna dada (primera descriptiva: 'description')
UCMCatalog2024_Splunk_description_1 = UCMCatalog2024_Splunk
UCMCatalog2024_Splunk_description_1['techniques_from_description'] = UCMCatalog2024_Splunk_description_1['DESCRIPTION'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Splunk_description_1 = UCMCatalog2024_Splunk_description_1[UCMCatalog2024_Splunk_description_1['techniques_from_description']!='']
UCMCatalog2024_Splunk_description_1 = UCMCatalog2024_Splunk_description_1.dropna(subset=['techniques_from_description'])
UCMCatalog2024_Splunk_description_1 = UCMCatalog2024_Splunk_description_1['techniques_from_description'].str.split(',').explode('techniques_from_description').reset_index()
UCMCatalog2024_Splunk_description_1.head(2)

,index,techniques_from_description


In [22]:
# Formateo para unión final
UCMCatalog2024_Splunk_description_1 = UCMCatalog2024_Splunk_description_1.assign(type_source='item description')[['techniques_from_description', 'type_source']]
UCMCatalog2024_Splunk_description_1 = UCMCatalog2024_Splunk_description_1[UCMCatalog2024_Splunk_description_1['techniques_from_description']!='']
UCMCatalog2024_Splunk_description_1.rename(columns={'techniques_from_description': 'techniques'}, inplace=True)
UCMCatalog2024_Splunk_description_1 = UCMCatalog2024_Splunk_description_1.drop_duplicates()
UCMCatalog2024_Splunk_description_1.head(2)

,techniques,type_source


In [23]:
# Obtención técnicas a partir de una columna dada (primera descriptiva: 'description')
UCMCatalog2024_Splunk_description_2 = UCMCatalog2024_Splunk
UCMCatalog2024_Splunk_description_2['techniques_from_description'] = UCMCatalog2024_Splunk_description_2['SEARCH'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Splunk_description_2 = UCMCatalog2024_Splunk_description_2[UCMCatalog2024_Splunk_description_2['techniques_from_description']!='']
UCMCatalog2024_Splunk_description_2 = UCMCatalog2024_Splunk_description_2.dropna(subset=['techniques_from_description'])
UCMCatalog2024_Splunk_description_2 = UCMCatalog2024_Splunk_description_2['techniques_from_description'].str.split(',').explode('techniques_from_description').reset_index()
UCMCatalog2024_Splunk_description_2.head(2)

,index,techniques_from_description


In [24]:
# Formateo para unión final
UCMCatalog2024_Splunk_description_2 = UCMCatalog2024_Splunk_description_2.assign(type_source='item description')[['techniques_from_description', 'type_source']]
UCMCatalog2024_Splunk_description_2 = UCMCatalog2024_Splunk_description_2[UCMCatalog2024_Splunk_description_2['techniques_from_description']!='']
UCMCatalog2024_Splunk_description_2.rename(columns={'techniques_from_description': 'techniques'}, inplace=True)
UCMCatalog2024_Splunk_description_2 = UCMCatalog2024_Splunk_description_2.drop_duplicates()
UCMCatalog2024_Splunk_description_2.head(2)

,techniques,type_source


In [25]:
# Unimos técnicas de UCMCatalog2024 Splunk
UCMCatalog2024_Splunk_union = pd.concat([UCMCatalog2024_Splunk_raw, UCMCatalog2024_Splunk_description_1, UCMCatalog2024_Splunk_description_2])
UCMCatalog2024_Splunk_union['techniques'] = UCMCatalog2024_Splunk_union['techniques'].str.strip()
UCMCatalog2024_Splunk_union = UCMCatalog2024_Splunk_union.drop_duplicates(subset=['techniques'])
UCMCatalog2024_Splunk_union = UCMCatalog2024_Splunk_union.sort_values(by='techniques')
UCMCatalog2024_Splunk_union.head()

,techniques,type_source
22,T1036,item raw
62,T1068,item raw
53,T1070,item raw
52,T1070.001,item raw
54,T1070.009,item raw


In [26]:
UCMCatalog2024_Splunk_union.shape

(33, 2)

In [27]:
UCMCatalog2024_Splunk_union.groupby('type_source').count()

,techniques
type_source,
item raw,33


In [28]:
# Guardado del fichero final
UCMCatalog2024_Splunk_union.to_csv('tecnhniques_UCMCatalog2024_Splunk.csv',sep=';',encoding='utf8',index=False)

### 3. UCM Catalog 2024 - QRADAR

In [29]:
UCMCatalog2024_Qradar_baseline = pd.read_excel(path_UCMCatalog2024_Qradar,'Baseline')
UCMCatalog2024_Qradar_baseline.head(3)

,Rule,Priority,Mitre Tactic,Mitre Technique,Log Source Type,Required Telemetry
0,[CyberProof] - [Windows] - Multiple Failed Log...,LOW,Credential Access,T1110 - Brute Force,Windows,NaN
1,[CyberProof] - [Windows] - Sensitive User Lock...,MED,Privilege Escalation,T1078 - Valid Accounts,Windows,Event ID 4740
2,[CyberProof] - [Windows] - Sensitive User pass...,MED,Privilege Escalation,T1078 - Valid Accounts,Windows,Event ID 4724


In [30]:
UCMCatalog2024_Qradar_baseline['MITRE_TECHNIQUE'] = UCMCatalog2024_Qradar_baseline['Mitre Technique'].str.upper()

In [31]:
# Obtención técnicas a partir de una columna dada (columna dedicada a la técnica Mitre: 'Mitre Technique')
UCMCatalog2024_Qradar_baseline_raw = UCMCatalog2024_Qradar_baseline
UCMCatalog2024_Qradar_baseline_raw['techniques_from_raw'] = UCMCatalog2024_Qradar_baseline_raw['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Qradar_baseline_raw = UCMCatalog2024_Qradar_baseline_raw[UCMCatalog2024_Qradar_baseline_raw['techniques_from_raw']!='']
UCMCatalog2024_Qradar_baseline_raw = UCMCatalog2024_Qradar_baseline_raw.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_baseline_raw = UCMCatalog2024_Qradar_baseline_raw['techniques_from_raw'].str.split(',').explode('techniques_from_raw').reset_index()
UCMCatalog2024_Qradar_baseline_raw.head(2)

,index,techniques_from_raw
0,0,T1110
1,1,T1078


In [32]:
# Formateo para unión final
UCMCatalog2024_Qradar_baseline_raw = UCMCatalog2024_Qradar_baseline_raw.assign(type_source='item raw')[['techniques_from_raw', 'type_source']]
UCMCatalog2024_Qradar_baseline_raw = UCMCatalog2024_Qradar_baseline_raw[UCMCatalog2024_Qradar_baseline_raw['techniques_from_raw']!='']
UCMCatalog2024_Qradar_baseline_raw.rename(columns={'techniques_from_raw': 'techniques'}, inplace=True)
UCMCatalog2024_Qradar_baseline_raw = UCMCatalog2024_Qradar_baseline_raw.drop_duplicates()
UCMCatalog2024_Qradar_baseline_raw.head(2)

,techniques,type_source
0,T1110,item raw
1,T1078,item raw


In [33]:
UCMCatalog2024_Qradar_baseline_raw.shape

(70, 2)

In [34]:
# Siguiente hoja del libro: ATOMICS
UCMCatalog2024_Qradar_atomics = pd.read_excel(path_UCMCatalog2024_Qradar,'Atomics')
UCMCatalog2024_Qradar_atomics.head(3)

,TREND,Rule,Priority,Mitre Tactic,Mitre Technique,Log Source Type
0,2024 - Ransomware,[AtomicRedTeam] - [InhibitSystemRecovery #1] ...,HIGH,Impact,T1490 - Inhibit System Recovery,Windows/Sysmon/Powershell
1,2024 - Ransomware,[AtomicRedTeam] - [InhibitSystemRecovery #2] ...,HIGH,Impact,T1490 - Inhibit System Recovery,Windows/Sysmon/Powershell
2,2024 - Ransomware,[AtomicRedTeam] - [InhibitSystemRecovery #3] ...,HIGH,Impact,T1490 - Inhibit System Recovery,Windows/Sysmon/Powershell


In [35]:
UCMCatalog2024_Qradar_atomics['MITRE_TECHNIQUE'] = UCMCatalog2024_Qradar_atomics['Mitre Technique'].str.upper()

In [36]:
# Obtención técnicas a partir de una columna dada (columna dedicada a la técnica Mitre: 'Mitre Technique')
UCMCatalog2024_Qradar_atomics_raw = UCMCatalog2024_Qradar_atomics
UCMCatalog2024_Qradar_atomics_raw['techniques_from_raw'] = UCMCatalog2024_Qradar_atomics_raw['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Qradar_atomics_raw = UCMCatalog2024_Qradar_atomics_raw[UCMCatalog2024_Qradar_atomics_raw['techniques_from_raw']!='']
UCMCatalog2024_Qradar_atomics_raw = UCMCatalog2024_Qradar_atomics_raw.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_atomics_raw = UCMCatalog2024_Qradar_atomics_raw['techniques_from_raw'].str.split(',').explode('techniques_from_raw').reset_index()
UCMCatalog2024_Qradar_atomics_raw.head(2)

,index,techniques_from_raw
0,0,T1490
1,1,T1490


In [37]:
# Formateo para unión final
UCMCatalog2024_Qradar_atomics_raw = UCMCatalog2024_Qradar_atomics_raw.assign(type_source='item raw')[['techniques_from_raw', 'type_source']]
UCMCatalog2024_Qradar_atomics_raw = UCMCatalog2024_Qradar_atomics_raw[UCMCatalog2024_Qradar_atomics_raw['techniques_from_raw']!='']
UCMCatalog2024_Qradar_atomics_raw.rename(columns={'techniques_from_raw': 'techniques'}, inplace=True)
UCMCatalog2024_Qradar_atomics_raw = UCMCatalog2024_Qradar_atomics_raw.drop_duplicates()
UCMCatalog2024_Qradar_atomics_raw.head(2)

,techniques,type_source
0,T1490,item raw
10,T1482,item raw


In [38]:
UCMCatalog2024_Qradar_atomics_raw.shape

(22, 2)

In [39]:
# Siguiente hoja del libro: FW
UCMCatalog2024_Qradar_fw= pd.read_excel(path_UCMCatalog2024_Qradar,'FW')
UCMCatalog2024_Qradar_fw.head(3)

,Rule,Priority,Mitre Tactic,Mitre Technique
0,[CyberProof] - [Fortigate] - R2L SSH Connection,Medium,NaN,NaN
1,[CyberProof] - [Fortigate] - R2L RDP Connection,Medium,NaN,NaN
2,[CyberProof] - [Fortigate] - WAF Alert Not-Blo...,Medium,NaN,NaN


In [40]:
UCMCatalog2024_Qradar_fw['MITRE_TECHNIQUE'] = UCMCatalog2024_Qradar_fw['Mitre Technique'].str.upper()

In [41]:
# Obtención técnicas a partir de una columna dada (columna dedicada a la técnica Mitre: 'Mitre Technique')
UCMCatalog2024_Qradar_fw_raw = UCMCatalog2024_Qradar_fw
UCMCatalog2024_Qradar_fw_raw['techniques_from_raw'] = UCMCatalog2024_Qradar_fw_raw['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Qradar_fw_raw = UCMCatalog2024_Qradar_fw_raw[UCMCatalog2024_Qradar_fw_raw['techniques_from_raw']!='']
UCMCatalog2024_Qradar_fw_raw = UCMCatalog2024_Qradar_fw_raw.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_fw_raw = UCMCatalog2024_Qradar_fw_raw['techniques_from_raw'].str.split(',').explode('techniques_from_raw').reset_index()
UCMCatalog2024_Qradar_fw_raw.head(2)

,index,techniques_from_raw
0,0,T1078
1,1,T1070


In [42]:
# Formateo para unión final
UCMCatalog2024_Qradar_fw_raw = UCMCatalog2024_Qradar_fw_raw.assign(type_source='item raw')[['techniques_from_raw', 'type_source']]
UCMCatalog2024_Qradar_fw_raw = UCMCatalog2024_Qradar_fw_raw[UCMCatalog2024_Qradar_fw_raw['techniques_from_raw']!='']
UCMCatalog2024_Qradar_fw_raw.rename(columns={'techniques_from_raw': 'techniques'}, inplace=True)
UCMCatalog2024_Qradar_fw_raw = UCMCatalog2024_Qradar_fw_raw.drop_duplicates()
UCMCatalog2024_Qradar_fw_raw.head(2)

,techniques,type_source
0,T1078,item raw
1,T1070,item raw


In [43]:
UCMCatalog2024_Qradar_fw_raw.shape

(7, 2)

In [44]:
# Siguiente hoja del libro: AES
UCMCatalog2024_Qradar_aws= pd.read_excel(path_UCMCatalog2024_Qradar,'AWS')
UCMCatalog2024_Qradar_aws.head(3)

,Rule,Priority,Mitre Tactic,Mitre Technique,Log Source Type
0,EC: AWS Cloud - Detected A Successful Login Fr...,MED,NaN,NaN,AWS
1,[CyberProof] - [AWS] - Config Disabling Channe...,HIGH,Defense Evasion,T1562.001 - Impair Defenses: Disable or Modify...,AWS
2,[CyberProof] - [AWS] - EC2 Startup Shell Scrip...,HIGH,Execution,T1059.001 - Command and Scripting Interpreter:...,AWS


In [45]:
UCMCatalog2024_Qradar_aws['MITRE_TECHNIQUE'] = UCMCatalog2024_Qradar_aws['Mitre Technique'].str.upper()

In [46]:
# Obtención técnicas a partir de una columna dada (columna dedicada a la técnica Mitre: 'Mitre Technique')
UCMCatalog2024_Qradar_aws_raw = UCMCatalog2024_Qradar_aws
UCMCatalog2024_Qradar_aws_raw['techniques_from_raw'] = UCMCatalog2024_Qradar_aws_raw['MITRE_TECHNIQUE'].apply(lambda x: find_techniques(x))
UCMCatalog2024_Qradar_aws_raw = UCMCatalog2024_Qradar_aws_raw[UCMCatalog2024_Qradar_aws_raw['techniques_from_raw']!='']
UCMCatalog2024_Qradar_aws_raw = UCMCatalog2024_Qradar_aws_raw.dropna(subset=['techniques_from_raw'])
UCMCatalog2024_Qradar_aws_raw = UCMCatalog2024_Qradar_aws_raw['techniques_from_raw'].str.split(',').explode('techniques_from_raw').reset_index()
UCMCatalog2024_Qradar_aws_raw.head(2)

,index,techniques_from_raw
0,0,T1562.001
1,1,T1562


In [47]:
# Formateo para unión final
UCMCatalog2024_Qradar_aws_raw = UCMCatalog2024_Qradar_aws_raw.assign(type_source='item raw')[['techniques_from_raw', 'type_source']]
UCMCatalog2024_Qradar_aws_raw = UCMCatalog2024_Qradar_aws_raw[UCMCatalog2024_Qradar_aws_raw['techniques_from_raw']!='']
UCMCatalog2024_Qradar_aws_raw.rename(columns={'techniques_from_raw': 'techniques'}, inplace=True)
UCMCatalog2024_Qradar_aws_raw = UCMCatalog2024_Qradar_aws_raw.drop_duplicates()
UCMCatalog2024_Qradar_aws_raw.head(2)

,techniques,type_source
0,T1562.001,item raw
1,T1562,item raw


In [48]:
UCMCatalog2024_Qradar_aws_raw.shape

(11, 2)

In [49]:
# Unimos técnicas de UCMCatalog2024 Qradar
UCMCatalog2024_Qradar_union = pd.concat([UCMCatalog2024_Qradar_baseline_raw, UCMCatalog2024_Qradar_atomics_raw, UCMCatalog2024_Qradar_fw_raw, UCMCatalog2024_Qradar_aws_raw])
UCMCatalog2024_Qradar_union['techniques'] = UCMCatalog2024_Qradar_union['techniques'].str.strip()
UCMCatalog2024_Qradar_union = UCMCatalog2024_Qradar_union.drop_duplicates(subset=['techniques'])
UCMCatalog2024_Qradar_union = UCMCatalog2024_Qradar_union.sort_values(by='techniques')
UCMCatalog2024_Qradar_union.head()

,techniques,type_source
32,T1003,item raw
105,T1003.006,item raw
11,T1020,item raw
63,T1021,item raw
76,T1021.001,item raw


In [50]:
UCMCatalog2024_Qradar_union.to_csv('tecnhniques_UCMCatalog2024_Qradar.csv',sep=';',encoding='utf8',index=False)

### 4. Unión final

In [51]:
UCMCatalog2024_Sentinel_union['source'] = 'Sentinel'

In [52]:
UCMCatalog2024_Splunk_union['source'] = 'Splunk'

In [53]:
UCMCatalog2024_Qradar_union['source'] = 'Qradar'

In [58]:
UCMCatalog2024 = pd.concat([UCMCatalog2024_Sentinel_union, UCMCatalog2024_Splunk_union, UCMCatalog2024_Qradar_union])
UCMCatalog2024 = UCMCatalog2024.sort_values(by='techniques')
UCMCatalog2024.to_csv('tecnhniques_UCMCatalog2024.csv',sep=';',encoding='utf8',index=False)

In [64]:
UCMCatalog2024_Sentinel_union.groupby(['type_source']).count()

,techniques,source
type_source,,
item description,3,3
item raw,221,221


In [67]:
UCMCatalog2024_Splunk_union.groupby(['type_source']).count()

,techniques,source
type_source,,
item raw,33,33


In [66]:
UCMCatalog2024_Qradar_union.groupby(['type_source']).count()

,techniques,source
type_source,,
item raw,76,76


In [63]:
UCMCatalog2024.groupby(['source','type_source']).count()

techniques
source   type_source                 
Qradar   item raw                  76
Sentinel item description           3
         item raw                 221
Splunk   item raw                  33

In [88]:
add_info = techniques_info(lift)
add_info.head(2)

[taxii2client.common] [WARNING ] [2024-05-16 13:05:02,134] TAXII Server Response did not include 'Content-Range' header - results could be incomplete


,technique_id,type,url,technique,technique_description,tactic,data_sources
0,T1059.010,attack-pattern,https://attack.mitre.org/techniques/T1059/010,AutoHotKey & AutoIT,Adversaries may execute commands and perform m...,[execution],"[Process: Process Creation, Command: Command E..."
1,T1564.012,attack-pattern,https://attack.mitre.org/techniques/T1564/012,File/Path Exclusions,Adversaries may attempt to hide their file-bas...,[defense-evasion],[File: File Creation]


In [93]:
UCMCatalog2024_info.shape

(333, 3)

In [91]:
UCMCatalog2024_info.head(2)

,techniques,type_source,source,type,url,technique,technique_description,tactic,data_sources
0,T0802,item raw,Sentinel,NaN,NaN,NaN,NaN,NaN,NaN
1,T0807,item raw,Sentinel,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
UCMCatalog2024_info = pd.merge(UCMCatalog2024, add_info, left_on='techniques', right_on='technique_id', how='left')
UCMCatalog2024_info = UCMCatalog2024_info.drop(columns='technique_id')

In [96]:
UCMCatalog2024_info.to_csv('tecnhniques_UCMCatalog2024_info.csv',sep=';',encoding='utf8',index=False,quoting=csv.QUOTE_ALL)